In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

df_train = pd.read_csv("train.csv/train.csv")
df_test = pd.read_csv("test.csv/test.csv")
df_test_labels = pd.read_csv("test_labels.csv/test_labels.csv")
df_sample_submission = pd.read_csv("sample_submission.csv/sample_submission.csv")

df_train.head()

In [ ]:
import html
import re

def preprocess_text(text):
    """Clean a comment while keeping wording, negation, and punctuation."""
    if pd.isna(text):
        return ""

    text = html.unescape(str(text))
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b", " EMAIL ", text)
    text = re.sub(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", " IPADDRESS ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

# Keep the original text; use clean_text as the model's text input.
df_train["clean_text"] = df_train["comment_text"].apply(preprocess_text)
df_test["clean_text"] = df_test["comment_text"].apply(preprocess_text)

df_train[["comment_text", "clean_text"]].head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# Select the comment text as the input features for the model.
X = df_train["clean_text"]

# Select the binary toxicity label that the model will learn to predict.
# A value of 0 means non-toxic, and a value of 1 means toxic.
y = df_train["toxic"]

# Split the labeled data into a training set and a validation set.
# The model learns from 80% of the comments and is evaluated on the other 20%.
# A fixed random_state makes the split reproducible.
# Stratification keeps the toxic/non-toxic proportions similar in both sets.
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create a Bag of Words vectorizer that counts words in each comment.
# It learns the vocabulary only from training comments to prevent data leakage.
bow_vectorizer = CountVectorizer()

# Learn the vocabulary and convert each training comment into a word-count vector.
X_train_bow = bow_vectorizer.fit_transform(X_train)

# Convert validation comments using the same vocabulary learned from training.
X_valid_bow = bow_vectorizer.transform(X_valid)

# Show the number of comments and vocabulary terms in each matrix.
print("Training matrix:", X_train_bow.shape)
print("Validation matrix:", X_valid_bow.shape)

# Report how many unique terms the vectorizer learned from training comments.
print("Vocabulary size:", len(bow_vectorizer.get_feature_names_out()))